# 01. Pivot index

**Tier:** Deep. **Why you'd use this:** You buy once and sell once, and the sale is on a later day. Walk the prices.

**After this lesson you can:** compare left and right totals with a prefix array; find an index where both sides match.
About 20 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does prefix[i] usually mean?
2. How do you get a left-side total from a prefix array?

<details><summary>Answers</summary>

For 01 pivot lesson: skim [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb) if needed.
</details>


## Recognize it

You need an index where the sum on the left equals the sum on the right. The prompt says "pivot index", "balance point", or "left sum equals right sum".

Signals: one index splits the list, and both sides are aggregate totals. A look-alike that is not this pattern: best buy-and-sell profit — that tracks a running minimum price, not equal side sums.

Another look-alike: "find any index where the value equals the average of neighbors." That is a local three-cell check, not a whole-side balance. Pivot cares about every cell to the left versus every cell to the right.


## Why it works this way

A nested check at index `i` sums everything left of `i` and everything right of `i`. Doing that for every `i` rescans most of the list each time, so the work grows like `n²`. The prefix insight is that the whole list has one total. If you know the sum of the left side while you walk, the right side is `total - left - nums[i]`. Equality of left and right becomes one comparison per index after a single total pass (or while you accumulate the total in a separate structure).

You can also build an explicit prefix array where `prefix[i]` is the sum of the first `i` values. Then left of index `i` is `prefix[i]` and right is `prefix[n] - prefix[i + 1]`. The running-left version uses constant extra memory; the array version makes many later range questions cheap. Both rest on the same identity: a range sum is a difference of two prefixes.

The pivot value itself belongs to neither side. Including it in `left` or in `right` is the usual silent bug. Empty sides are allowed: index `0` has left sum `0`, and the last index has right sum `0`. Returning `-1` when no pivot exists is the sibling's empty answer.

Design-wise, prefix totals trade a linear preprocess for constant-time side checks. That is the same reason range-sum queries precompute prefixes. Stock-style problems reuse a running extreme instead; do not force a pivot array onto a min-price scan. Name the balance question before you reach for either tool.

Mechanically, after the total pass you never need another full rescan: each step adds `nums[i]` into `left` only after the comparison, so the next index sees the updated left. That discipline — compare, then absorb — keeps the sides honest and the second pass linear.


## When to use it

Use a pivot / balance check when left and right aggregates must match at one index. Use a running minimum (or maximum) when the question is about best difference over time. Use an explicit prefix array when you will ask many range questions after one build. Prefer a nested rescan only for tiny lists.

If both sides must match in length rather than in sum, you are counting indexes, not totals — a midpoint formula is enough and prefix sums are overkill.


## Brute force, and why it's slow

At every index, sum the left side and the right side from scratch.


In [1]:
def pivot_index_slow(nums):
    additions = 0
    for index in range(len(nums)):
        left = 0
        for i in range(index):
            left += nums[i]
            additions += 1
        right = 0
        for i in range(index + 1, len(nums)):
            right += nums[i]
            additions += 1
        if left == right:
            return index, additions
    return -1, additions


print(pivot_index_slow([1, 7, 3, 6, 5, 6]))
print(pivot_index_slow([1, 2, 3]))


(3, 20)
(-1, 6)


## Key insight

If `total` is the sum of the list and `left` is the sum before index `i`, the right sum is `total - left - nums[i]`. Compare those two without rescanning. Leave `nums[i]` out of both sides.


## Worked example

Return an index whose left sum equals its right sum, or `-1` when none exists.


In [2]:
def pivot_index(nums: list[int]) -> int:
    # 1. One total for the whole list.
    total = sum(nums)
    left = 0
    for index, value in enumerate(nums):
        # 2. Right is whatever is left after removing left and the pivot value.
        if left == total - left - value:
            return index
        left += value
    return -1


print(pivot_index([1, 7, 3, 6, 5, 6]))
print(pivot_index([1, 2, 3]))
print(pivot_index([2, 1, -1]))


3
-1
0


## Trace

`[1, 7, 3, 6, 5, 6]`, `total = 28`:

| index | value | left | right (`28 - left - value`) | equal? |
|---|---|---|---|---|
| 0 | 1 | 0 | 27 | no |
| 1 | 7 | 1 | 20 | no |
| 2 | 3 | 8 | 17 | no |
| 3 | 6 | 11 | 11 | **yes → 3** |


## Cost

One pass to total and one pass to walk: time grows with `n`. Extra memory is constant for the running-left form. The nested left/right sums pay about `n² / 2` additions. An explicit prefix array raises memory to `n` slots in exchange for easier later range math.


## Variants

Not solved here: [Best Time to Buy and Sell Stock](../../challenges/problems/20-best-time-to-buy-and-sell-stock.ipynb), [Subarray Sum Equals K](../../challenges/problems/43-subarray-sum-equals-k.ipynb), [Maximum Subarray](../../challenges/problems/21-maximum-subarray.ipynb).


## Common mistake

This bug is silent. The comparison uses `total - left` and forgets to remove the pivot value, so the right side is too large by `nums[i]`. There is no traceback. The print is `-1`, and the answer is `3`.


In [3]:
def pivot_index_wrong(nums):
    total = sum(nums)
    left = 0
    for index, value in enumerate(nums):
        if left == total - left:  # pivot still counted on the right
            return index
        left += value
    return -1


print(pivot_index_wrong([1, 7, 3, 6, 5, 6]))


-1


The fix compares `left` to `total - left - value`. The pivot belongs to neither side. The buggy cell prints `-1` for `[1, 7, 3, 6, 5, 6]` because it counts the pivot on one side.


In [4]:
print(pivot_index([1, 7, 3, 6, 5, 6]))


3


## Python idioms

`enumerate` walks index and value together. `sum(nums)` is the total preprocess; for huge streams you would accumulate total in the same pass with a second structure, but random-access lists make the two-pass form clear.


In [5]:
def pivot_index_prefix(nums: list[int]) -> int:
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1] + value)
    total = prefix[-1]
    for index in range(len(nums)):
        left = prefix[index]
        right = total - prefix[index + 1]
        if left == right:
            return index
    return -1


print(pivot_index_prefix([1, 7, 3, 6, 5, 6]))


3


## Recap

- Right sum is total minus left minus the pivot value.
- Empty sides are allowed.
- One linear walk beats rescanning both sides at every index.

Next: [practice](01-pivot-practice.ipynb).
